# SCAMPER demo

End-to-end demo: imitation 3×3 grid → RGM 3×3 (logvar=−4) → motion prior 3×3 (logvar=−4 and 0) → walk-rear (SCAMPER, MIMIC-MJX, scratch) → arena-forage (SCAMPER, MIMIC-MJX, scratch).

For every section: a SETUP cell builds the env/policy, then an optional **LOAD** cell reads a saved rollout (skip the ROLLOUT cell if you load), a **ROLLOUT** cell computes fresh, a **SAVE** cell writes h5, a **RENDER** cell writes mp4, and a **GC** cell frees JAX/MuJoCo state.

Rollouts → `/home/mila/a/aidan.sirbu/SCAMPER/data/demos/rollouts`.
Videos → `/home/mila/a/aidan.sirbu/SCAMPER/data/demos/videos`.

**Before running**: fill in the checkpoint paths in the global config cell — the task-transfer sections will fail until those are valid.


In [ ]:
import os

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import functools
import gc
import json
from pathlib import Path

import h5py
import imageio
import jax
import jax.numpy as jnp
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.animation as manim
import mujoco
import numpy as np
import orbax.checkpoint as ocp
from IPython.display import Video, display
from jax import random
from mujoco import mjx
from omegaconf import OmegaConf

from brax.training import distribution
from brax.training.acme import running_statistics, specs
from brax.training.agents.ppo import networks as ppo_networks

from scamper.agent.distill.teacher import load_frozen_teacher
from scamper.agent.observation_utils import flatten_obs_dict, normalize_dict_obs
from scamper.analysis import rollout as scamper_rollout
from scamper.environment.reference_clips import ReferenceClips
from scamper.environment.rodent import consts as rodent_consts
from scamper.environment.utils import _recolour_tree, dm_scale_spec
from scamper.scripts.collect_rollout_dataset import (
    create_environment as create_distill_prior_env,
    create_prior_policy_fn,
    fix_config_paths,
)
from scamper.scripts.collect_rollout_dataset_distill import (
    create_rgm_policy_fn,
    load_distill_prior_checkpoint_for_rollouts,
)

In [ ]:
# ============ STORAGE PATHS ============
DEMO_ROOT = Path("/home/mila/a/aidan.sirbu/SCAMPER/data/demos")
ROLLOUTS_DIR = DEMO_ROOT / "rollouts"
VIDEOS_DIR = DEMO_ROOT / "videos"
ROLLOUTS_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# ============ SHARED CLIP / GRID / RENDER CONFIG ============
CLIP_INDICES = [0, 2, 3, 4, 5, 6, 8, 11, 12]
NUM_CLIPS = len(CLIP_INDICES)
GRID_ROWS = 3
GRID_COLS = 3
GRID_SPACING = 0.3  # meters between adjacent grid cells (xy)
GHOST_ALPHA = 0.3  # ghost transparency for the imitation grid

GRID_CAM_AZIMUTH = -135.0
GRID_CAM_ELEVATION = -21.0
GRID_CAM_DISTANCE = 1.9
GRID_CAM_LOOKAT = (0.04, 0.04, 0.0)

GRID_LIGHT_DIR = (0.0, 0.0, -1.0)
GRID_LIGHT_AMBIENT = (0.05, 0.05, 0.05)
GRID_LIGHT_DIFFUSE = (0.7, 0.7, 0.7)
GRID_LIGHT_SPECULAR = (0.3, 0.3, 0.3)

RENDER_HEIGHT = 720
RENDER_WIDTH = 1280
RENDER_FPS = 50

# Rodent dataset
REFERENCE_DATA_PATH = (
    "/home/mila/a/aidan.sirbu/SCAMPER/data/rodent/rodent_reference_clips.h5"
)
CLIP_LENGTH = 250
RESCALE_FACTOR = 0.9  # rodent default
WALKER_XML = str(rodent_consts.RODENT_NO_TAIL_COLLISION_XML)
ARENA_XML = str(rodent_consts.ARENA_XML_PATH)
SINGLE_RODENT_CAMERA = "close_profile-rodent"

# Rollout length for upstream sections
NUM_ROLLOUT_STEPS = 1000
SEED = 42

# ============ CHECKPOINTS — FILL IN ============
# Section 1 (imitation): a distill checkpoint whose teacher is the imitation policy.
IMITATION_DISTILL_CHECKPOINT = (
    "/home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_checkpoints/260327_134740_079934"
)
IMITATION_DISTILL_CHECKPOINT_STEP = None  # None = latest
IMITATION_TEACHER_CHECKPOINT_PATH = (
    None  # None = use the path baked into the distill cfg
)
IMITATION_TEACHER_CHECKPOINT_STEP = None

# Sections 2/3/4 (RGM + motion prior): a distill_prior checkpoint that contains rgm + prior + decoder.
DISTILL_PRIOR_CHECKPOINT = "/home/mila/a/aidan.sirbu/SCAMPER/scratch/distill_prior_checkpoints/260406_130524_381481"
DISTILL_PRIOR_CHECKPOINT_STEP = 54  # None = latest

# Sections 5/6/7 (walk-rear). Modes: "distill", "decoder_only", "scratch".
WALKREAR_SCAMPER_CHECKPOINT = ""  # FILL IN — mode=distill
WALKREAR_SCAMPER_STEP = None
WALKREAR_MIMICMJX_CHECKPOINT = ""  # FILL IN — mode=decoder_only
WALKREAR_MIMICMJX_STEP = None
WALKREAR_SCRATCH_CHECKPOINT = ""  # FILL IN — mode=scratch
WALKREAR_SCRATCH_STEP = None
WALKREAR_NUM_STEPS = 500

# Sections 8/9/10 (arena-forage). Modes: "distill_prior", "decoder_only", "scratch".
ARENAFORAGE_SCAMPER_CHECKPOINT = ""  # FILL IN — mode=distill_prior
ARENAFORAGE_SCAMPER_STEP = None
ARENAFORAGE_MIMICMJX_CHECKPOINT = ""  # FILL IN — mode=decoder_only
ARENAFORAGE_MIMICMJX_STEP = None
ARENAFORAGE_SCRATCH_CHECKPOINT = ""  # FILL IN — mode=scratch
ARENAFORAGE_SCRATCH_STEP = None
ARENAFORAGE_NUM_STEPS = 1000

print(f"Demo root: {DEMO_ROOT}")
print(f"  rollouts -> {ROLLOUTS_DIR}")
print(f"  videos   -> {VIDEOS_DIR}")
print(f"Clip indices: {CLIP_INDICES}")

In [ ]:
# ============ SHARED HELPERS ============


def save_rollout_h5(path, qpos_dict, attrs=None, extra_groups=None):
    """Save {clip_idx: ndarray} (or any int-keyed dict) to h5 with file-level attrs."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        for k, v in (attrs or {}).items():
            try:
                f.attrs[k] = v
            except TypeError:
                f.attrs[k] = json.dumps(v)
        for k, arr in qpos_dict.items():
            f.create_dataset(f"clip_{int(k)}", data=np.asarray(arr), compression="gzip")
        for group_name, group_dict in (extra_groups or {}).items():
            grp = f.create_group(group_name)
            for k, arr in group_dict.items():
                grp.create_dataset(
                    f"clip_{int(k)}", data=np.asarray(arr), compression="gzip"
                )
    print(f"saved {len(qpos_dict)} entries to {path}")
    return path


def load_rollout_h5(path, extra_group_names=()):
    """Return ({clip_idx: ndarray}, attrs_dict, {group_name: {clip_idx: ndarray}})."""
    qpos_dict = {}
    attrs = {}
    extras = {name: {} for name in extra_group_names}
    with h5py.File(path, "r") as f:
        for k, v in f.attrs.items():
            attrs[k] = v
        for key in f.keys():
            if key.startswith("clip_"):
                qpos_dict[int(key[len("clip_") :])] = f[key][:]
        for name in extra_group_names:
            if name in f:
                for key in f[name].keys():
                    if key.startswith("clip_"):
                        extras[name][int(key[len("clip_") :])] = f[name][key][:]
    print(f"loaded {len(qpos_dict)} entries from {path}")
    return qpos_dict, attrs, extras


def release(*names):
    """Delete named globals, then gc.collect() and jax.clear_caches()."""
    for n in names:
        if n in globals():
            try:
                del globals()[n]
            except KeyError:
                pass
    gc.collect()
    try:
        jax.clear_caches()
    except Exception:
        pass
    print(f"released {list(names)}; jax live arrays: {len(jax.live_arrays())}")


def grid_xy_offsets(rows=GRID_ROWS, cols=GRID_COLS, spacing=GRID_SPACING):
    """xy offsets centered at the origin, row-major."""
    return np.array(
        [
            (
                (c - (cols - 1) / 2.0) * spacing,
                (r - (rows - 1) / 2.0) * spacing,
            )
            for r in range(rows)
            for c in range(cols)
        ],
        dtype=np.float32,
    )


def _strip_lights(spec):
    def visit(body):
        for light in list(body.lights):
            spec.delete(light)
        for child in body.bodies:
            visit(child)

    visit(spec.worldbody)


def build_grid_scene(n_real, n_ghost, ghost_alpha=GHOST_ALPHA):
    """Build an MjModel containing n_real opaque rodents + n_ghost transparent rodents.

    Returns (model, data, nq_single, real_xy, ghost_xy, colors).
    """
    n_total = n_real + n_ghost
    real_xy = grid_xy_offsets()
    ghost_xy = grid_xy_offsets()
    colors = [tuple(cm.tab10(i % 10)) for i in range(max(n_real, n_ghost, 1))]

    spec = mujoco.MjSpec.from_file(ARENA_XML)
    spec.worldbody.add_light(
        name="grid_light",
        type=int(mujoco.mjtLightType.mjLIGHT_DIRECTIONAL),
        dir=list(GRID_LIGHT_DIR),
        ambient=list(GRID_LIGHT_AMBIENT),
        diffuse=list(GRID_LIGHT_DIFFUSE),
        specular=list(GRID_LIGHT_SPECULAR),
    )
    for i in range(n_real):
        rod = mujoco.MjSpec.from_file(WALKER_XML)
        if RESCALE_FACTOR != 1.0:
            rod = dm_scale_spec(rod, RESCALE_FACTOR)
        _strip_lights(rod)
        for body in rod.worldbody.bodies:
            _recolour_tree(body, rgba=[*colors[i][:3], 1.0])
        frame = spec.worldbody.add_frame(
            pos=(float(real_xy[i, 0]), float(real_xy[i, 1]), 0.0)
        )
        body = frame.attach_body(rod.worldbody, "", suffix=f"-r{i}")
        body.add_freejoint()
    for i in range(n_ghost):
        rod = mujoco.MjSpec.from_file(WALKER_XML)
        if RESCALE_FACTOR != 1.0:
            rod = dm_scale_spec(rod, RESCALE_FACTOR)
        _strip_lights(rod)
        for body in rod.worldbody.bodies:
            _recolour_tree(body, rgba=[*colors[i][:3], ghost_alpha])
        frame = spec.worldbody.add_frame(
            pos=(float(ghost_xy[i, 0]), float(ghost_xy[i, 1]), 0.0)
        )
        body = frame.attach_body(rod.worldbody, "", suffix=f"-g{i}")
        body.add_freejoint()

    model = spec.compile()
    model.vis.global_.offwidth = max(model.vis.global_.offwidth, RENDER_WIDTH)
    model.vis.global_.offheight = max(model.vis.global_.offheight, RENDER_HEIGHT)
    data = mujoco.MjData(model)
    nq_single = model.nq // max(n_total, 1)
    return model, data, nq_single, real_xy, ghost_xy, colors


def recenter_to_grid(qpos_array, grid_xy):
    """Shift each trajectory so its initial xy lands on its grid cell.

    qpos_array: [N, T, nq]. grid_xy: [N, 2].
    Returns a copy with qpos[..., 0:2] adjusted.
    """
    out = qpos_array.copy()
    for i in range(qpos_array.shape[0]):
        start_xy = qpos_array[i, 0, :2]
        out[i, :, 0] += grid_xy[i, 0] - start_xy[0]
        out[i, :, 1] += grid_xy[i, 1] - start_xy[1]
    return out


def render_grid_frames(model, data, nq_single, qpos_real, qpos_ghost=None, cam=None):
    """Render frames of the grid scene given per-rodent qpos sequences.

    qpos_real: [N_real, T, nq]; qpos_ghost: [N_ghost, T, nq] or None.
    Returns list of HxWx3 uint8 frames.
    """
    n_real = qpos_real.shape[0]
    T = qpos_real.shape[1]
    n_ghost = 0 if qpos_ghost is None else qpos_ghost.shape[0]

    if cam is None:
        cam = mujoco.MjvCamera()
        cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        cam.azimuth = GRID_CAM_AZIMUTH
        cam.elevation = GRID_CAM_ELEVATION
        cam.distance = GRID_CAM_DISTANCE
        cam.lookat[:] = GRID_CAM_LOOKAT

    renderer = mujoco.Renderer(model, height=RENDER_HEIGHT, width=RENDER_WIDTH)
    frames = []
    try:
        for t in range(T):
            for i in range(n_real):
                data.qpos[i * nq_single : (i + 1) * nq_single] = qpos_real[i, t]
            for i in range(n_ghost):
                data.qpos[(n_real + i) * nq_single : (n_real + i + 1) * nq_single] = (
                    qpos_ghost[i, t]
                )
            mujoco.mj_kinematics(model, data)
            mujoco.mj_comPos(model, data)
            renderer.update_scene(data, camera=cam)
            frames.append(renderer.render().copy())
    finally:
        renderer.close()
    return frames


def pad_frames_for_codec(frames):
    pad = (-frames[0].shape[1]) % 16
    if pad:
        frames = [
            np.pad(f, ((0, 0), (0, pad), (0, 0)), constant_values=255) for f in frames
        ]
    return frames


def write_video(frames, path, fps=RENDER_FPS):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = pad_frames_for_codec(frames)
    imageio.mimsave(str(path), frames, fps=fps, quality=8)
    print(f"saved video to {path} ({len(frames)} frames)")
    return path


def stack_grid_qpos(rollout_dict, clip_indices):
    """Stack a {clip_idx: [T, nq]} dict in the order given by clip_indices."""
    return np.stack([np.asarray(rollout_dict[c]) for c in clip_indices], axis=0)


def setup_imitation_env(
    distill_checkpoint,
    distill_step,
    teacher_checkpoint=None,
    teacher_step=None,
    mujoco_impl="warp",
    xml_variant_path=None,
):
    """Load a distill checkpoint, restore its teacher, and build the imitation env.

    Returns (im_env, reference_clips, teacher_fn, teacher_normalizer, teacher_cfg, distill_cfg, action_dist).
    """
    print(f"Reading distill cfg: {distill_checkpoint}")
    options = ocp.CheckpointManagerOptions(create=False, step_prefix="DistillNetwork")
    with ocp.CheckpointManager(distill_checkpoint, options=options) as mgr:
        step = mgr.latest_step() if distill_step is None else distill_step
        cfg_dict = mgr.restore(
            step, args=ocp.args.Composite(config=ocp.args.JsonRestore())
        )["config"]
    distill_cfg = OmegaConf.create(cfg_dict)

    t_path = (
        teacher_checkpoint
        if teacher_checkpoint is not None
        else distill_cfg.teacher_config.checkpoint_path
    )
    t_step = (
        teacher_step
        if teacher_step is not None
        else distill_cfg.teacher_config.get("checkpoint_step", None)
    )
    print(f"Loading frozen teacher: {t_path} (step={t_step})")
    teacher_fn, teacher_normalizer, teacher_cfg, _ = load_frozen_teacher(
        t_path, step=t_step
    )

    OmegaConf.set_struct(distill_cfg.env_config, False)
    OmegaConf.update(distill_cfg.env_config, "reference_data_path", REFERENCE_DATA_PATH)
    if mujoco_impl is not None:
        OmegaConf.update(distill_cfg.env_config, "mujoco_impl", mujoco_impl)
    if xml_variant_path is not None:
        OmegaConf.update(
            distill_cfg.env_config, "walker_xml_path", str(xml_variant_path)
        )
    OmegaConf.set_struct(distill_cfg.env_config, True)

    im_env = scamper_rollout.create_environment(distill_cfg)

    # Unwrap to get the reference_clips object.
    unwrapped = im_env
    while hasattr(unwrapped, "_env"):
        unwrapped = unwrapped._env
    reference_clips = unwrapped.reference_clips

    action_size = teacher_cfg["network_config"]["action_size"]
    action_dist = distribution.NormalTanhDistribution(event_size=action_size)

    print(
        f"  env: {type(im_env).__name__}; clip_length={distill_cfg.env_config.clip_length}; "
        f"mujoco_impl={distill_cfg.env_config.mujoco_impl}"
    )
    return (
        im_env,
        reference_clips,
        teacher_fn,
        teacher_normalizer,
        teacher_cfg,
        distill_cfg,
        action_dist,
    )


def setup_distill_prior_env(checkpoint, step):
    """Load a distill_prior checkpoint and build its rollout env. Returns
    (env, reference_clips, cfg, policy_params)."""
    cfg, policy_params = load_distill_prior_checkpoint_for_rollouts(checkpoint, step)
    OmegaConf.set_struct(cfg.env_config, False)
    OmegaConf.update(cfg.env_config, "mujoco_impl", "warp")
    OmegaConf.update(
        cfg.env_config,
        "walker_xml_path",
        str(rodent_consts.RODENT_NO_TAIL_COLLISION_XML),
    )
    OmegaConf.update(cfg.env_config, "reference_data_path", REFERENCE_DATA_PATH)
    OmegaConf.set_struct(cfg.env_config, True)
    cfg = fix_config_paths(cfg)
    env, reference_clips = create_distill_prior_env(cfg)
    print(f"Loaded distill_prior {Path(checkpoint).name} step={step}")
    return env, reference_clips, cfg, policy_params


def make_clip_initial_state(env, reference_clips, rng_key, clip_idx):
    """Reset env, then overwrite qpos with reference clip's frame-0 qpos and zero qvel."""
    state = env.reset(rng_key)
    qpos0 = reference_clips.qpos[clip_idx, 0]
    qpos0 = jnp.asarray(qpos0)
    data = state.data.replace(qpos=qpos0, qvel=jnp.zeros(env.mjx_model.nv))
    data = mjx.forward(env.mjx_model, data)
    return state.replace(data=data)


def run_proprio_rollout(env, policy_fn, starting_state, num_steps, rng_key):
    """Generic scan-based rollout loop where policy_fn(proprio, key) -> (action, _)."""
    jit_step = jax.jit(env.step)

    def step_fn(carry, _):
        state, key = carry
        key, action_key = random.split(key)
        if isinstance(state.obs, dict) or hasattr(state.obs, "get"):
            proprio = flatten_obs_dict(state.obs)["proprioception"]
        else:
            proprio = state.obs
        action, _ = policy_fn(proprio, action_key)
        next_state = jit_step(state, action)
        return (next_state, key), next_state

    (_, _), all_states = jax.lax.scan(
        step_fn, (starting_state, rng_key), None, length=num_steps
    )
    jax.tree_util.tree_map(lambda x: x.block_until_ready(), all_states)
    return all_states


def rollout_distill_prior_grid(
    env, reference_clips, policy_fn, clip_indices, num_steps, seed=SEED
):
    """Run policy_fn on each clip's start-pose; return [N, T, nq] qpos array."""
    qpos_by_clip = {}
    for ci in clip_indices:
        rng = random.PRNGKey(seed + int(ci))
        rng_init, rng_rollout = random.split(rng)
        start = make_clip_initial_state(env, reference_clips, rng_init, ci)
        all_states = run_proprio_rollout(env, policy_fn, start, num_steps, rng_rollout)
        qpos_by_clip[ci] = np.asarray(all_states.data.qpos)
        print(f"  clip {ci}: qpos {qpos_by_clip[ci].shape}")
    return qpos_by_clip

---

## Section 1 — Imitation 3×3 grid (teacher rollout)

Roll out the frozen imitation teacher on the 9 reference clips in `CLIP_INDICES`. The grid render shows 9 opaque rodents (rolled-out teacher) with their reference-clip ghosts overlaid in the matching colour at `GHOST_ALPHA`.

Files: `imitation_grid.h5` (with `qpos` and `qpos_ref` groups), `imitation_grid.mp4`.


In [ ]:
# Build imitation env, load frozen teacher, prepare reference clips.
(
    im_env,
    im_reference_clips,
    im_teacher_fn,
    im_teacher_normalizer,
    im_teacher_cfg,
    im_distill_cfg,
    im_action_dist,
) = setup_imitation_env(
    IMITATION_DISTILL_CHECKPOINT,
    IMITATION_DISTILL_CHECKPOINT_STEP,
    teacher_checkpoint=IMITATION_TEACHER_CHECKPOINT_PATH,
    teacher_step=IMITATION_TEACHER_CHECKPOINT_STEP,
    mujoco_impl="warp",
    xml_variant_path=rodent_consts.RODENT_NO_TAIL_COLLISION_XML,
)

# Episode length = (clip_length - reference_length) * steps_per_frame.
_im_mocap_hz = float(im_distill_cfg.env_config.mocap_hz)
_im_steps_per_frame = int((1.0 / _im_mocap_hz) / im_distill_cfg.env_config.ctrl_dt)
IM_EPISODE_LENGTH = (
    im_distill_cfg.env_config.clip_length - im_distill_cfg.env_config.reference_length
) * _im_steps_per_frame
print(f"Imitation episode length: {IM_EPISODE_LENGTH} steps; clips: {NUM_CLIPS}")

In [ ]:
# OPTIONAL: load existing imitation rollout from disk and skip the next (rollout) cell.
_im_load_path = ROLLOUTS_DIR / "imitation_grid.h5"
_im_qpos, _im_attrs, _im_extras = load_rollout_h5(
    _im_load_path, extra_group_names=("qpos_ref",)
)
im_grid_qpos = stack_grid_qpos(_im_qpos, CLIP_INDICES)
im_grid_qpos_ref = stack_grid_qpos(_im_extras["qpos_ref"], CLIP_INDICES)
print(
    f"loaded im_grid_qpos {im_grid_qpos.shape}; im_grid_qpos_ref {im_grid_qpos_ref.shape}"
)

In [ ]:
# Roll out the frozen teacher across CLIP_INDICES (vmap'd JIT).
def _im_teacher_action(obs):
    flat_obs = flatten_obs_dict(obs)
    norm_obs = normalize_dict_obs(flat_obs, im_teacher_normalizer)
    logits = im_teacher_fn(norm_obs)
    ctrl = im_action_dist.mode(logits)
    return jnp.squeeze(ctrl, axis=0) if ctrl.ndim == 2 else ctrl


def _im_step_fn(carry, _):
    state, act_rng = carry
    act_rng, _sub = jax.random.split(act_rng)
    time_in_frames = state.data.time * _im_mocap_hz
    frame = jnp.floor(time_in_frames + state.info["start_frame"]).astype(int)
    ref = im_reference_clips.at(clip=state.info["reference_clip"], frame=frame)
    ctrl = _im_teacher_action(state.obs)
    next_state = im_env.step(state, ctrl)
    return (next_state, act_rng), (state.data.qpos, ref.qpos)


def _im_rollout_one(clip_idx):
    rng = jax.random.PRNGKey(SEED)
    reset_key, act_key = jax.random.split(rng)
    state = im_env.reset(reset_key, clip_idx=clip_idx, start_frame=jnp.int32(0))
    init_ref = im_reference_clips.at(clip=clip_idx, frame=jnp.int32(0))
    (_, _), (qpos_seq, ref_seq) = jax.lax.scan(
        _im_step_fn, (state, act_key), None, length=IM_EPISODE_LENGTH
    )
    qpos_full = jnp.concatenate([state.data.qpos[None], qpos_seq], axis=0)
    ref_full = jnp.concatenate([init_ref.qpos[None], ref_seq], axis=0)
    return {"qpos": qpos_full, "qpos_ref": ref_full}


_im_clip_idxs = jnp.asarray(CLIP_INDICES, dtype=jnp.int32)
print("JIT-compiling vmap rollout (slow first time)...")
_im_rollout_vmap = jax.jit(jax.vmap(_im_rollout_one))
_im_data = _im_rollout_vmap(_im_clip_idxs)
im_grid_qpos = np.array(_im_data["qpos"])
im_grid_qpos_ref = np.array(_im_data["qpos_ref"])
print(
    f"Done. im_grid_qpos {im_grid_qpos.shape}; im_grid_qpos_ref {im_grid_qpos_ref.shape}"
)

In [ ]:
# Save imitation rollout (qpos + qpos_ref groups) to h5.
_im_qpos_dict = {ci: im_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)}
_im_ref_dict = {ci: im_grid_qpos_ref[i] for i, ci in enumerate(CLIP_INDICES)}
save_rollout_h5(
    ROLLOUTS_DIR / "imitation_grid.h5",
    _im_qpos_dict,
    attrs={
        "kind": "imitation_teacher",
        "clip_indices": np.asarray(CLIP_INDICES),
        "episode_length": int(IM_EPISODE_LENGTH),
        "distill_checkpoint": str(IMITATION_DISTILL_CHECKPOINT),
        "seed": int(SEED),
    },
    extra_groups={"qpos_ref": _im_ref_dict},
)

In [ ]:
# Render the 3x3 grid: real rodents (opaque) + reference ghosts (transparent), each
# starting at its grid cell (xy is re-centered).
_im_model, _im_data_grid, _im_nq_single, _im_real_xy, _im_ghost_xy, _im_colors = (
    build_grid_scene(n_real=NUM_CLIPS, n_ghost=NUM_CLIPS, ghost_alpha=GHOST_ALPHA)
)
print(f"  grid model nq={_im_model.nq} (per-rodent nq={_im_nq_single})")

_im_real_q = recenter_to_grid(im_grid_qpos, _im_real_xy)
_im_ghost_q = recenter_to_grid(im_grid_qpos_ref, _im_ghost_xy)

_im_frames = render_grid_frames(
    _im_model,
    _im_data_grid,
    _im_nq_single,
    qpos_real=_im_real_q,
    qpos_ghost=_im_ghost_q,
)
_im_video = write_video(_im_frames, VIDEOS_DIR / "imitation_grid.mp4", fps=RENDER_FPS)
display(Video(str(_im_video), embed=True))

In [ ]:
release(
    "im_env",
    "im_reference_clips",
    "im_teacher_fn",
    "im_teacher_normalizer",
    "im_teacher_cfg",
    "im_distill_cfg",
    "im_action_dist",
    "im_grid_qpos",
    "im_grid_qpos_ref",
    "_im_rollout_vmap",
    "_im_data",
    "_im_clip_idxs",
    "_im_qpos_dict",
    "_im_ref_dict",
    "_im_model",
    "_im_data_grid",
    "_im_nq_single",
    "_im_real_xy",
    "_im_ghost_xy",
    "_im_colors",
    "_im_real_q",
    "_im_ghost_q",
    "_im_frames",
    "_im_video",
    "_im_qpos",
    "_im_attrs",
    "_im_extras",
)

---

## Section 2 — RGM 3×3 grid at logvar = −4

Roll out the RGM (Residual Generative Model) network from the distill_prior checkpoint at fixed `logvar=-4`, initialized from the qpos at frame 0 of each clip in `CLIP_INDICES`.

Files: `rgm_logvar-4.h5`, `rgm_logvar-4.mp4`.


In [ ]:
# Load distill_prior env + policy params, build RGM policy at logvar=-4.
rgm_env, rgm_reference_clips, rgm_cfg, rgm_policy_params = setup_distill_prior_env(
    DISTILL_PRIOR_CHECKPOINT, DISTILL_PRIOR_CHECKPOINT_STEP
)
rgm_policy_fn = create_rgm_policy_fn(
    rgm_policy_params, rgm_cfg, fixed_logvar=-4.0, deterministic=False
)
print("RGM policy ready (logvar=-4)")

In [ ]:
# OPTIONAL: load existing RGM rollout from disk and skip the next (rollout) cell.
_rgm_load_path = ROLLOUTS_DIR / "rgm_logvar-4.h5"
_rgm_qpos, _rgm_attrs, _ = load_rollout_h5(_rgm_load_path)
rgm_grid_qpos = stack_grid_qpos(_rgm_qpos, CLIP_INDICES)
print(f"loaded rgm_grid_qpos {rgm_grid_qpos.shape}")

In [ ]:
# Roll out RGM from each clip's frame-0 pose.
_rgm_qpos_by_clip = rollout_distill_prior_grid(
    rgm_env,
    rgm_reference_clips,
    rgm_policy_fn,
    CLIP_INDICES,
    NUM_ROLLOUT_STEPS,
    seed=SEED,
)
rgm_grid_qpos = stack_grid_qpos(_rgm_qpos_by_clip, CLIP_INDICES)
print(f"rgm_grid_qpos {rgm_grid_qpos.shape}")

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "rgm_logvar-4.h5",
    {ci: rgm_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
    attrs={
        "kind": "rgm",
        "logvar": -4.0,
        "clip_indices": np.asarray(CLIP_INDICES),
        "num_steps": int(NUM_ROLLOUT_STEPS),
        "distill_prior_checkpoint": str(DISTILL_PRIOR_CHECKPOINT),
        "step": (
            -1
            if DISTILL_PRIOR_CHECKPOINT_STEP is None
            else int(DISTILL_PRIOR_CHECKPOINT_STEP)
        ),
        "seed": int(SEED),
    },
)

In [ ]:
# Render the RGM 3x3 grid (no ghosts — just the rolled-out rodents).
_rgm_model, _rgm_data_grid, _rgm_nq_single, _rgm_real_xy, _, _ = build_grid_scene(
    n_real=NUM_CLIPS, n_ghost=0
)
print(f"  grid model nq={_rgm_model.nq} (per-rodent nq={_rgm_nq_single})")
_rgm_real_q = recenter_to_grid(rgm_grid_qpos, _rgm_real_xy)
_rgm_frames = render_grid_frames(
    _rgm_model, _rgm_data_grid, _rgm_nq_single, qpos_real=_rgm_real_q
)
_rgm_video = write_video(_rgm_frames, VIDEOS_DIR / "rgm_logvar-4.mp4", fps=RENDER_FPS)
display(Video(str(_rgm_video), embed=True))

In [ ]:
release(
    "rgm_env",
    "rgm_reference_clips",
    "rgm_cfg",
    "rgm_policy_params",
    "rgm_policy_fn",
    "rgm_grid_qpos",
    "_rgm_qpos_by_clip",
    "_rgm_model",
    "_rgm_data_grid",
    "_rgm_nq_single",
    "_rgm_real_xy",
    "_rgm_real_q",
    "_rgm_frames",
    "_rgm_video",
    "_rgm_qpos",
    "_rgm_attrs",
)

---

## Section 3 — Motion prior 3×3 grid at logvar = −4

Roll out the prior network from the distill_prior checkpoint at fixed `logvar=-4`.

Files: `motion_prior_logvar-4.h5`, `motion_prior_logvar-4.mp4`.


In [ ]:
prior4_env, prior4_reference_clips, prior4_cfg, prior4_policy_params = (
    setup_distill_prior_env(DISTILL_PRIOR_CHECKPOINT, DISTILL_PRIOR_CHECKPOINT_STEP)
)
prior4_policy_fn = create_prior_policy_fn(
    prior4_policy_params, prior4_cfg, fixed_logvar=-4.0, deterministic=False
)
print("Motion prior policy ready (logvar=-4)")

In [ ]:
_p4_load_path = ROLLOUTS_DIR / "motion_prior_logvar-4.h5"
_p4_qpos, _p4_attrs, _ = load_rollout_h5(_p4_load_path)
prior4_grid_qpos = stack_grid_qpos(_p4_qpos, CLIP_INDICES)
print(f"loaded prior4_grid_qpos {prior4_grid_qpos.shape}")

In [ ]:
_p4_qpos_by_clip = rollout_distill_prior_grid(
    prior4_env,
    prior4_reference_clips,
    prior4_policy_fn,
    CLIP_INDICES,
    NUM_ROLLOUT_STEPS,
    seed=SEED,
)
prior4_grid_qpos = stack_grid_qpos(_p4_qpos_by_clip, CLIP_INDICES)
print(f"prior4_grid_qpos {prior4_grid_qpos.shape}")

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "motion_prior_logvar-4.h5",
    {ci: prior4_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
    attrs={
        "kind": "motion_prior",
        "logvar": -4.0,
        "clip_indices": np.asarray(CLIP_INDICES),
        "num_steps": int(NUM_ROLLOUT_STEPS),
        "distill_prior_checkpoint": str(DISTILL_PRIOR_CHECKPOINT),
        "step": (
            -1
            if DISTILL_PRIOR_CHECKPOINT_STEP is None
            else int(DISTILL_PRIOR_CHECKPOINT_STEP)
        ),
        "seed": int(SEED),
    },
)

In [ ]:
_p4_model, _p4_data_grid, _p4_nq_single, _p4_real_xy, _, _ = build_grid_scene(
    n_real=NUM_CLIPS, n_ghost=0
)
print(f"  grid model nq={_p4_model.nq} (per-rodent nq={_p4_nq_single})")
_p4_real_q = recenter_to_grid(prior4_grid_qpos, _p4_real_xy)
_p4_frames = render_grid_frames(
    _p4_model, _p4_data_grid, _p4_nq_single, qpos_real=_p4_real_q
)
_p4_video = write_video(
    _p4_frames, VIDEOS_DIR / "motion_prior_logvar-4.mp4", fps=RENDER_FPS
)
display(Video(str(_p4_video), embed=True))

In [ ]:
release(
    "prior4_env",
    "prior4_reference_clips",
    "prior4_cfg",
    "prior4_policy_params",
    "prior4_policy_fn",
    "prior4_grid_qpos",
    "_p4_qpos_by_clip",
    "_p4_model",
    "_p4_data_grid",
    "_p4_nq_single",
    "_p4_real_xy",
    "_p4_real_q",
    "_p4_frames",
    "_p4_video",
    "_p4_qpos",
    "_p4_attrs",
)

---

## Section 4 — Motion prior 3×3 grid at logvar = 0

Same as Section 3 but with `logvar=0` (random sampling, much higher entropy).

Files: `motion_prior_logvar0.h5`, `motion_prior_logvar0.mp4`.


In [ ]:
prior0_env, prior0_reference_clips, prior0_cfg, prior0_policy_params = (
    setup_distill_prior_env(DISTILL_PRIOR_CHECKPOINT, DISTILL_PRIOR_CHECKPOINT_STEP)
)
prior0_policy_fn = create_prior_policy_fn(
    prior0_policy_params, prior0_cfg, fixed_logvar=0.0, deterministic=False
)
print("Motion prior policy ready (logvar=0)")

In [ ]:
_p0_load_path = ROLLOUTS_DIR / "motion_prior_logvar0.h5"
_p0_qpos, _p0_attrs, _ = load_rollout_h5(_p0_load_path)
prior0_grid_qpos = stack_grid_qpos(_p0_qpos, CLIP_INDICES)
print(f"loaded prior0_grid_qpos {prior0_grid_qpos.shape}")

In [ ]:
_p0_qpos_by_clip = rollout_distill_prior_grid(
    prior0_env,
    prior0_reference_clips,
    prior0_policy_fn,
    CLIP_INDICES,
    NUM_ROLLOUT_STEPS,
    seed=SEED,
)
prior0_grid_qpos = stack_grid_qpos(_p0_qpos_by_clip, CLIP_INDICES)
print(f"prior0_grid_qpos {prior0_grid_qpos.shape}")

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "motion_prior_logvar0.h5",
    {ci: prior0_grid_qpos[i] for i, ci in enumerate(CLIP_INDICES)},
    attrs={
        "kind": "motion_prior",
        "logvar": 0.0,
        "clip_indices": np.asarray(CLIP_INDICES),
        "num_steps": int(NUM_ROLLOUT_STEPS),
        "distill_prior_checkpoint": str(DISTILL_PRIOR_CHECKPOINT),
        "step": (
            -1
            if DISTILL_PRIOR_CHECKPOINT_STEP is None
            else int(DISTILL_PRIOR_CHECKPOINT_STEP)
        ),
        "seed": int(SEED),
    },
)

In [ ]:
_p0_model, _p0_data_grid, _p0_nq_single, _p0_real_xy, _, _ = build_grid_scene(
    n_real=NUM_CLIPS, n_ghost=0
)
print(f"  grid model nq={_p0_model.nq} (per-rodent nq={_p0_nq_single})")
_p0_real_q = recenter_to_grid(prior0_grid_qpos, _p0_real_xy)
_p0_frames = render_grid_frames(
    _p0_model, _p0_data_grid, _p0_nq_single, qpos_real=_p0_real_q
)
_p0_video = write_video(
    _p0_frames, VIDEOS_DIR / "motion_prior_logvar0.mp4", fps=RENDER_FPS
)
display(Video(str(_p0_video), embed=True))

In [ ]:
release(
    "prior0_env",
    "prior0_reference_clips",
    "prior0_cfg",
    "prior0_policy_params",
    "prior0_policy_fn",
    "prior0_grid_qpos",
    "_p0_qpos_by_clip",
    "_p0_model",
    "_p0_data_grid",
    "_p0_nq_single",
    "_p0_real_xy",
    "_p0_real_q",
    "_p0_frames",
    "_p0_video",
    "_p0_qpos",
    "_p0_attrs",
)

---

## Walk-rear / arena-forage shared helpers

The next six sections all use the same task-transfer rollout pattern:
1. Load checkpoint metadata to discover `mode`.
2. Load frozen networks (decoder + optionally prior).
3. Build the task env with the appropriate wrapper for `mode`.
4. Build the PPO inference fn.
5. Roll out one episode and collect qpos.


In [ ]:
# Walk-rear helpers.
from scamper.environment.rodent import walk_rear
from scamper.agent.task_transfer.walk_rear.checkpoint_utils import (
    load_prior_checkpoint as wr_load_prior_checkpoint,
    load_task_transfer_metadata as wr_load_task_transfer_metadata,
    load_task_transfer_brax_params as wr_load_task_transfer_brax_params,
    make_decoder_inference_fn as wr_make_decoder_inference_fn,
    make_prior_inference_fn as wr_make_prior_inference_fn,
)
from scamper.agent.task_transfer.walk_rear.wrappers import (
    DecoderHighLevelWrapper as WRDecoderWrapper,
    FlatObsWrapper as WRFlatObsWrapper,
    PriorDecoderHighLevelWrapper as WRPriorDecoderWrapper,
)


def _wr_xml_path(xml_variant):
    if xml_variant == "limited-collisions":
        return rodent_consts.RODENT_XML_PATH
    if xml_variant == "full-collisions":
        return rodent_consts.RODENT_FULL_COLLISION_XML
    if xml_variant == "no-tail-collisions":
        return rodent_consts.RODENT_NO_TAIL_COLLISION_XML
    raise ValueError(f"Unknown xml_variant: {xml_variant!r}")


def _wr_build_env_cfg(config):
    env_cfg = walk_rear.default_config()
    env_cfg.walker_xml_path = _wr_xml_path(
        config["env"].get("xml_variant", "full-collisions")
    )
    env_cfg.mujoco_impl = "jax"
    env_cfg.ctrl_dt = config["checkpoint_config"]["env_config"]["ctrl_dt"]
    env_cfg.torque_actuators = True
    for key in (
        "target_x",
        "target_y",
        "target_z",
        "xy_tolerance",
        "z_tolerance",
        "hold_duration",
        "distance_margin",
    ):
        if key in config["env"]:
            setattr(env_cfg, key, config["env"][key])
    if "reward_weights" in config["env"]:
        env_cfg.reward_terms = {
            name: {"weight": w} for name, w in config["env"]["reward_weights"].items()
        }
    return env_cfg


def setup_walkrear(checkpoint_path, step):
    """Build a wrapped walk_rear env + jit'd inference fn for a task-transfer checkpoint.

    Returns (env, base_env, jit_step, jit_inference_fn, inference_params, start_state, config, mode).
    """
    info = wr_load_task_transfer_metadata(checkpoint_path, step=step)
    config = info["config"]
    resolved_step = info["step"]
    ckpt_cfg = config["checkpoint_config"]
    mode = config["mode"]
    print(f"Walk-rear mode={mode!r}, step={resolved_step}")

    env_cfg = _wr_build_env_cfg(config)
    base_env = walk_rear.WalkRear(config=env_cfg)

    if mode == "distill":
        encoder_p, prior_p, decoder_p, normalizer_p, _ = wr_load_prior_checkpoint(
            config["checkpoint"]["path"],
            step=config["checkpoint"]["step"],
            step_prefix="DistillNetwork",
        )
        latent_size = ckpt_cfg["network_config"]["intention_size"]
        proprio_size = ckpt_cfg["network_config"]["obs_sizes"]["proprioception"]
        decoder_fn = wr_make_decoder_inference_fn(decoder_p, normalizer_p, ckpt_cfg)
        prior_fn = wr_make_prior_inference_fn(prior_p, normalizer_p, ckpt_cfg)
        env = WRPriorDecoderWrapper(
            base_env,
            prior_fn,
            decoder_fn,
            latent_size,
            proprio_size,
            deterministic_prior=config["distill"]["deterministic_prior"],
            noise_logvar=config["distill"]["noise_logvar"],
        )
    elif mode == "decoder_only":
        _, _, decoder_p, normalizer_p, _ = wr_load_prior_checkpoint(
            config["checkpoint"]["path"],
            step=config["checkpoint"]["step"],
            step_prefix="DistillNetwork",
        )
        latent_size = ckpt_cfg["network_config"]["intention_size"]
        proprio_size = ckpt_cfg["network_config"]["obs_sizes"]["proprioception"]
        decoder_fn = wr_make_decoder_inference_fn(decoder_p, normalizer_p, ckpt_cfg)
        env = WRDecoderWrapper(base_env, decoder_fn, latent_size, proprio_size)
    elif mode == "scratch":
        env = WRFlatObsWrapper(base_env)
    else:
        raise ValueError(f"Unknown walk-rear mode: {mode!r}")

    jit_reset = jax.jit(env.reset)
    jit_step = jax.jit(env.step)
    rng = jax.random.PRNGKey(0)
    start_state = jit_reset(rng)

    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        policy_hidden_layer_sizes=tuple(config["network"]["policy_layers"]),
        value_hidden_layer_sizes=tuple(config["network"]["value_layers"]),
    )
    ppo_network = network_factory(
        start_state.obs.shape[-1],
        env.action_size,
        preprocess_observations_fn=running_statistics.normalize,
    )
    key_p, key_v = jax.random.split(jax.random.PRNGKey(0))
    dummy_normalizer = running_statistics.init_state(
        specs.Array(start_state.obs.shape[-1:], jnp.dtype("float32"))
    )
    dummy_policy = ppo_network.policy_network.init(key_p)
    dummy_value = ppo_network.value_network.init(key_v)
    abstract_brax_params = (dummy_normalizer, dummy_policy, dummy_value)
    brax_params = wr_load_task_transfer_brax_params(
        checkpoint_path=checkpoint_path,
        abstract_brax_params=abstract_brax_params,
        step=resolved_step,
    )
    normalizer_params_ppo, policy_params, _ = brax_params
    policy_network = ppo_network.policy_network
    parametric_action_distribution = ppo_network.parametric_action_distribution

    def deterministic_policy(params, observations, key_sample):
        del key_sample
        logits = policy_network.apply(params[0], params[1], observations)
        return parametric_action_distribution.mode(logits), {}

    jit_inference_fn = jax.jit(deterministic_policy)
    inference_params = (normalizer_params_ppo, policy_params)
    return (
        env,
        base_env,
        jit_step,
        jit_inference_fn,
        inference_params,
        start_state,
        config,
        mode,
    )


def task_transfer_rollout(
    start_state, jit_step, jit_inference_fn, inference_params, num_steps, seed
):
    """Single-episode rollout. Returns ndarray qpos[T+1, nq] (clipped at done)."""
    state = start_state
    states = [state]
    act_rng = jax.random.PRNGKey(seed + 1)
    done_step = None
    for t in range(num_steps):
        act_rng, key = jax.random.split(act_rng)
        action, _ = jit_inference_fn(inference_params, state.obs, key)
        state = jit_step(state, action)
        states.append(state)
        if bool(state.done) and done_step is None:
            done_step = t + 1
            print(f"Episode done at step {done_step}")
            break
    qpos = np.stack([np.asarray(s.data.qpos) for s in states], axis=0)
    return qpos, states


def render_single_rodent_qpos(
    base_env,
    qpos_traj,
    camera_name=SINGLE_RODENT_CAMERA,
    height=RENDER_HEIGHT,
    width=RENDER_WIDTH,
):
    """Render a single rodent through qpos_traj using its base mj_model and a named camera."""
    model = base_env.mj_model
    data = mujoco.MjData(model)
    cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, camera_name)
    cam_arg = cam_id if cam_id >= 0 else None
    if cam_arg is None:
        print(f"  (camera {camera_name!r} not found; using free camera)")
    renderer = mujoco.Renderer(model, height=height, width=width)
    frames = []
    try:
        for t in range(qpos_traj.shape[0]):
            data.qpos[:] = qpos_traj[t]
            mujoco.mj_forward(model, data)
            if cam_arg is None:
                renderer.update_scene(data)
            else:
                renderer.update_scene(data, camera=cam_arg)
            frames.append(renderer.render().copy())
    finally:
        renderer.close()
    return frames

---

## Section 5 — Walk-rear: SCAMPER (`mode=distill`)

Files: `walkrear_scamper.h5`, `walkrear_scamper.mp4`.


In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_SCAMPER_CHECKPOINT, WALKREAR_SCAMPER_STEP)
print(f"  obs_size={wr_start_state.obs.shape[-1]}, action_size={wr_env.action_size}")

In [ ]:
# OPTIONAL: load saved walkrear SCAMPER rollout (skip the next ROLLOUT cell if loaded).
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_scamper.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

In [ ]:
wr_qpos, wr_states = task_transfer_rollout(
    wr_start_state,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    WALKREAR_NUM_STEPS,
    seed=SEED,
)
print(
    f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
)

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "walkrear_scamper.h5",
    {0: wr_qpos},
    attrs={
        "kind": "walkrear",
        "variant": "SCAMPER",
        "mode": str(wr_mode),
        "checkpoint": str(WALKREAR_SCAMPER_CHECKPOINT),
        "step": -1 if WALKREAR_SCAMPER_STEP is None else int(WALKREAR_SCAMPER_STEP),
        "num_steps_requested": int(WALKREAR_NUM_STEPS),
        "num_steps_actual": int(wr_qpos.shape[0]),
        "seed": int(SEED),
    },
)

In [ ]:
_wr_frames = render_single_rodent_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(_wr_frames, VIDEOS_DIR / "walkrear_scamper.mp4", fps=RENDER_FPS)
display(Video(str(_wr_video), embed=True))

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

---

## Section 6 — Walk-rear: MIMIC-MJX (`mode=decoder_only`)

Files: `walkrear_mimicmjx.h5`, `walkrear_mimicmjx.mp4`.


In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_MIMICMJX_CHECKPOINT, WALKREAR_MIMICMJX_STEP)

In [ ]:
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_mimicmjx.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

In [ ]:
wr_qpos, wr_states = task_transfer_rollout(
    wr_start_state,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    WALKREAR_NUM_STEPS,
    seed=SEED,
)
print(
    f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
)

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "walkrear_mimicmjx.h5",
    {0: wr_qpos},
    attrs={
        "kind": "walkrear",
        "variant": "MIMIC-MJX",
        "mode": str(wr_mode),
        "checkpoint": str(WALKREAR_MIMICMJX_CHECKPOINT),
        "step": -1 if WALKREAR_MIMICMJX_STEP is None else int(WALKREAR_MIMICMJX_STEP),
        "num_steps_requested": int(WALKREAR_NUM_STEPS),
        "num_steps_actual": int(wr_qpos.shape[0]),
        "seed": int(SEED),
    },
)

In [ ]:
_wr_frames = render_single_rodent_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(
    _wr_frames, VIDEOS_DIR / "walkrear_mimicmjx.mp4", fps=RENDER_FPS
)
display(Video(str(_wr_video), embed=True))

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

---

## Section 7 — Walk-rear: scratch (`mode=scratch`)

Files: `walkrear_scratch.h5`, `walkrear_scratch.mp4`.


In [ ]:
(
    wr_env,
    wr_base_env,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    wr_start_state,
    wr_config,
    wr_mode,
) = setup_walkrear(WALKREAR_SCRATCH_CHECKPOINT, WALKREAR_SCRATCH_STEP)

In [ ]:
_wr_qpos_dict, _wr_attrs, _ = load_rollout_h5(ROLLOUTS_DIR / "walkrear_scratch.h5")
wr_qpos = np.asarray(list(_wr_qpos_dict.values())[0])
print(f"loaded wr_qpos {wr_qpos.shape}")

In [ ]:
wr_qpos, wr_states = task_transfer_rollout(
    wr_start_state,
    wr_jit_step,
    wr_jit_inf,
    wr_inf_params,
    WALKREAR_NUM_STEPS,
    seed=SEED,
)
print(
    f"wr_qpos {wr_qpos.shape}; final torso xy={wr_qpos[-1, :2]}, z={wr_qpos[-1, 2]:.3f}"
)

In [ ]:
save_rollout_h5(
    ROLLOUTS_DIR / "walkrear_scratch.h5",
    {0: wr_qpos},
    attrs={
        "kind": "walkrear",
        "variant": "scratch",
        "mode": str(wr_mode),
        "checkpoint": str(WALKREAR_SCRATCH_CHECKPOINT),
        "step": -1 if WALKREAR_SCRATCH_STEP is None else int(WALKREAR_SCRATCH_STEP),
        "num_steps_requested": int(WALKREAR_NUM_STEPS),
        "num_steps_actual": int(wr_qpos.shape[0]),
        "seed": int(SEED),
    },
)

In [ ]:
_wr_frames = render_single_rodent_qpos(wr_base_env, wr_qpos)
_wr_video = write_video(_wr_frames, VIDEOS_DIR / "walkrear_scratch.mp4", fps=RENDER_FPS)
display(Video(str(_wr_video), embed=True))

In [ ]:
release(
    "wr_env",
    "wr_base_env",
    "wr_jit_step",
    "wr_jit_inf",
    "wr_inf_params",
    "wr_start_state",
    "wr_config",
    "wr_mode",
    "wr_qpos",
    "wr_states",
    "_wr_frames",
    "_wr_video",
    "_wr_qpos_dict",
    "_wr_attrs",
)

---

## Arena-forage shared helpers


In [ ]:
# Arena-forage helpers.
from scamper.environment.rodent import arena_forage as af_env
from scamper.agent.task_transfer.arena_forage.checkpoint_utils import (
    load_prior_checkpoint as af_load_prior_checkpoint,
    load_distill_prior_checkpoint as af_load_distill_prior_checkpoint,
    load_task_transfer_metadata as af_load_task_transfer_metadata,
    load_task_transfer_brax_params as af_load_task_transfer_brax_params,
    make_decoder_inference_fn as af_make_decoder_inference_fn,
    make_prior_inference_fn as af_make_prior_inference_fn,
    make_rgm_inference_fn as af_make_rgm_inference_fn,
)
from scamper.agent.task_transfer.arena_forage.wrappers import (
    DecoderHighLevelWrapper as AFDecoderWrapper,
    FlatObsWrapper as AFFlatObsWrapper,
    PriorDecoderHighLevelWrapper as AFPriorDecoderWrapper,
    PriorRGMDecoderHighLevelWrapper as AFPriorRGMDecoderWrapper,
    RGMDecoderHighLevelWrapper as AFRGMDecoderWrapper,
)


def _af_xml_path(xml_variant):
    if xml_variant == "limited-collisions":
        return rodent_consts.RODENT_XML_PATH
    if xml_variant == "full-collisions":
        return rodent_consts.RODENT_FULL_COLLISION_XML
    if xml_variant == "no-tail-collisions":
        return rodent_consts.RODENT_NO_TAIL_COLLISION_XML
    raise ValueError(f"Unknown xml_variant: {xml_variant!r}")


def _af_build_env_cfg(config):
    env_cfg = af_env.default_config()
    env_cfg.walker_xml_path = _af_xml_path(
        config["env"].get("xml_variant", "no-tail-collisions")
    )
    env_cfg.mujoco_impl = "jax"  # force jax for rollout
    env_cfg.ctrl_dt = config["checkpoint_config"]["env_config"]["ctrl_dt"]
    env_cfg.num_treats = config["env"]["num_treats"]
    env_cfg.treat_sphere_radius = config["env"]["treat_sphere_radius"]
    env_cfg.treat_height = config["env"]["treat_height"]
    env_cfg.min_spawn_radius = config["env"]["min_spawn_radius"]
    env_cfg.spawn_radius = config["env"]["spawn_radius"]
    env_cfg.scent_sigma = config["env"]["scent_sigma"]
    env_cfg.reward_terms = {
        "treat_collection": {
            "weight": config["env"]["reward_weights"]["treat_collection"]
        },
        "scent_proximity": {
            "weight": config["env"]["reward_weights"]["scent_proximity"]
        },
    }
    if config["mode"] == "scratch":
        env_cfg.torque_actuators = config.get("scratch", {}).get(
            "torque_actuators", True
        )
    else:
        env_cfg.torque_actuators = True
    return env_cfg


def setup_arena_forage(checkpoint_path, step):
    """Build a wrapped arena_forage env + jit'd inference fn.

    Returns (env, base_env, jit_step, jit_inference_fn, inference_params, start_state, config, mode).
    """
    info = af_load_task_transfer_metadata(checkpoint_path, step=step)
    config = info["config"]
    resolved_step = info["step"]
    ckpt_cfg = config["checkpoint_config"]
    mode = config["mode"]
    print(f"Arena-forage mode={mode!r}, step={resolved_step}")

    # Load frozen networks based on mode.
    if mode in ("distill_prior", "distill_prior_no_rgm", "rgm_only"):
        encoder_p, prior_p, decoder_p, rgm_p, normalizer_p, _ = (
            af_load_distill_prior_checkpoint(
                config["checkpoint"]["path"], config["checkpoint"]["step"]
            )
        )
    elif mode == "distill":
        encoder_p, prior_p, decoder_p, normalizer_p, _ = af_load_prior_checkpoint(
            config["checkpoint"]["path"],
            step=config["checkpoint"]["step"],
            step_prefix="DistillNetwork",
        )
        rgm_p = None
    elif mode in ("decoder_only", "prior_decoder"):
        encoder_p, prior_p, decoder_p, normalizer_p, _ = af_load_prior_checkpoint(
            config["checkpoint"]["path"],
            step=config["checkpoint"]["step"],
            step_prefix="PriorNetwork",
        )
        rgm_p = None
    elif mode == "scratch":
        encoder_p = prior_p = decoder_p = rgm_p = normalizer_p = None
    else:
        raise ValueError(f"Unknown arena-forage mode: {mode!r}")

    latent_size = ckpt_cfg["network_config"]["intention_size"] if ckpt_cfg else None
    proprio_size = (
        ckpt_cfg["network_config"]["obs_sizes"]["proprioception"] if ckpt_cfg else None
    )
    if mode != "scratch":
        decoder_fn = af_make_decoder_inference_fn(decoder_p, normalizer_p, ckpt_cfg)
    else:
        decoder_fn = None

    env_cfg = _af_build_env_cfg(config)
    base_env = af_env.ArenaForage(config=env_cfg)
    base_env._mjx_model = mjx.put_model(base_env._mj_model, impl=env_cfg.mujoco_impl)

    if mode == "decoder_only":
        env = AFDecoderWrapper(base_env, decoder_fn, latent_size, proprio_size)
    elif mode == "scratch":
        env = AFFlatObsWrapper(base_env)
    elif mode == "distill_prior":
        prior_fn = af_make_prior_inference_fn(prior_p, normalizer_p, ckpt_cfg)
        rgm_fn = af_make_rgm_inference_fn(rgm_p, normalizer_p, ckpt_cfg)
        mc = config["distill_prior"]
        env = AFPriorRGMDecoderWrapper(
            base_env,
            prior_fn,
            rgm_fn,
            decoder_fn,
            latent_size,
            proprio_size,
            gate_bias=mc["gate_bias"],
            gate_margin=mc.get("gate_margin", 0.0),
            rgm_noise=mc["rgm_noise"],
            rgm_logvar=mc.get("rgm_logvar", None),
        )
    elif mode == "rgm_only":
        rgm_fn = af_make_rgm_inference_fn(rgm_p, normalizer_p, ckpt_cfg)
        mc = config["rgm_only"]
        env = AFRGMDecoderWrapper(
            base_env,
            rgm_fn,
            decoder_fn,
            latent_size,
            proprio_size,
            rgm_noise=mc["rgm_noise"],
            rgm_logvar=mc.get("rgm_logvar", None),
        )
    elif mode in ("distill", "distill_prior_no_rgm", "prior_decoder"):
        prior_fn = af_make_prior_inference_fn(prior_p, normalizer_p, ckpt_cfg)
        mc = config[mode]
        env = AFPriorDecoderWrapper(
            base_env,
            prior_fn,
            decoder_fn,
            latent_size,
            proprio_size,
            deterministic_prior=mc["deterministic_prior"],
            noise_logvar=mc["noise_logvar"],
        )
    else:
        raise ValueError(f"Unhandled arena-forage mode: {mode!r}")

    jit_reset = jax.jit(env.reset)
    jit_step = jax.jit(env.step)
    rng = jax.random.PRNGKey(0)
    start_state = jit_reset(rng)

    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        policy_hidden_layer_sizes=tuple(config["network"]["policy_layers"]),
        value_hidden_layer_sizes=tuple(config["network"]["value_layers"]),
    )
    ppo_network = network_factory(
        start_state.obs.shape[-1],
        env.action_size,
        preprocess_observations_fn=running_statistics.normalize,
    )
    key_p, key_v = jax.random.split(jax.random.PRNGKey(0))
    dummy_normalizer = running_statistics.init_state(
        specs.Array(start_state.obs.shape[-1:], jnp.dtype("float32"))
    )
    dummy_policy = ppo_network.policy_network.init(key_p)
    dummy_value = ppo_network.value_network.init(key_v)
    abstract_brax_params = (dummy_normalizer, dummy_policy, dummy_value)
    brax_params = af_load_task_transfer_brax_params(
        checkpoint_path=checkpoint_path,
        abstract_brax_params=abstract_brax_params,
        step=resolved_step,
    )
    normalizer_params_ppo, policy_params, _ = brax_params
    policy_network = ppo_network.policy_network
    parametric_action_distribution = ppo_network.parametric_action_distribution

    def deterministic_policy(params, observations, key_sample):
        del key_sample
        logits = policy_network.apply(params[0], params[1], observations)
        return parametric_action_distribution.mode(logits), {}

    jit_inference_fn = jax.jit(deterministic_policy)
    inference_params = (normalizer_params_ppo, policy_params)
    return (
        env,
        base_env,
        jit_step,
        jit_inference_fn,
        inference_params,
        start_state,
        config,
        mode,
    )


def arena_forage_rollout(
    start_state, jit_step, jit_inference_fn, inference_params, num_steps, seed
):
    """Single-episode arena-forage rollout. Captures qpos and treat_collected per step.

    Returns dict with keys:
      qpos:             [T+1, nq]
      treat_positions:  [num_treats, 3]   (from initial state.info)
      treat_collected:  [T+1, num_treats]
      scent_sigma, spawn_radius           (env constants)
    """
    state = start_state
    states = [state]
    act_rng = jax.random.PRNGKey(seed + 1)
    done_step = None
    for t in range(num_steps):
        act_rng, key = jax.random.split(act_rng)
        action, _ = jit_inference_fn(inference_params, state.obs, key)
        state = jit_step(state, action)
        states.append(state)
        if bool(state.done) and done_step is None:
            done_step = t + 1
            print(f"Episode done at step {done_step}")
            break

    qpos = np.stack([np.asarray(s.data.qpos) for s in states], axis=0)
    treat_positions = np.asarray(states[0].info["treat_positions"])
    treat_collected = np.stack(
        [np.asarray(s.info["treat_collected"]) for s in states], axis=0
    )
    return {
        "qpos": qpos,
        "treat_positions": treat_positions,
        "treat_collected": treat_collected.astype(np.bool_),
    }, states


def save_arena_forage_rollout(path, rollout, attrs):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        for k, v in attrs.items():
            try:
                f.attrs[k] = v
            except TypeError:
                f.attrs[k] = json.dumps(v)
        f.create_dataset("qpos", data=np.asarray(rollout["qpos"]), compression="gzip")
        f.create_dataset("treat_positions", data=np.asarray(rollout["treat_positions"]))
        f.create_dataset(
            "treat_collected",
            data=np.asarray(rollout["treat_collected"]),
            compression="gzip",
        )
    print(f"saved arena-forage rollout to {path}")
    return path


def load_arena_forage_rollout(path):
    with h5py.File(path, "r") as f:
        rollout = {
            "qpos": f["qpos"][:],
            "treat_positions": f["treat_positions"][:],
            "treat_collected": f["treat_collected"][:],
        }
        attrs = {k: v for k, v in f.attrs.items()}
    print(f"loaded arena-forage rollout from {path}")
    return rollout, attrs


def render_arena_forage_3panel(
    base_env,
    rollout,
    scent_sigma,
    spawn_radius,
    out_path,
    fps=RENDER_FPS,
    grid_resolution=100,
    max_animation_frames=300,
    cam_azimuth=135.0,
    cam_elevation=-35.0,
    cam_distance=2.0,
    cam_lookat=(0.0, 0.0, 0.0),
    panel_height=720,
    panel_width=720,
):
    """Render an arena-forage rollout as a 3-panel animation:
    (A) 3D angled rodent render | (B) scent heatmap + xy trail | (C) 2D xy trace.
    """
    qpos = rollout["qpos"]  # [T, nq]
    treat_positions = rollout["treat_positions"]  # [N_treats, 3]
    treat_collected = rollout["treat_collected"]  # [T, N_treats] bool
    T = qpos.shape[0]

    stride = max(1, T // max_animation_frames)
    frames_idx = np.arange(0, T, stride)
    n_frames = len(frames_idx)

    # ---- Panel A: pre-render 3D frames once. ----
    model = base_env.mj_model
    data = mujoco.MjData(model)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.azimuth = cam_azimuth
    cam.elevation = cam_elevation
    cam.distance = cam_distance
    cam.lookat[:] = cam_lookat

    renderer = mujoco.Renderer(model, height=panel_height, width=panel_width)
    rgb_frames = []
    try:
        for t in frames_idx:
            data.qpos[:] = qpos[t]
            mujoco.mj_forward(model, data)
            renderer.update_scene(data, camera=cam)
            rgb_frames.append(renderer.render().copy())
    finally:
        renderer.close()

    # ---- Pre-compute scent grid bounds and a per-treat scent image. ----
    extent = float(spawn_radius) + 0.3
    xs = np.linspace(-extent, extent, grid_resolution)
    ys = np.linspace(-extent, extent, grid_resolution)
    gx, gy = np.meshgrid(xs, ys)
    per_treat_grid = np.zeros(
        (treat_positions.shape[0], grid_resolution, grid_resolution)
    )
    for i, (tx, ty, _tz) in enumerate(treat_positions):
        per_treat_grid[i] = np.exp(
            -((gx - tx) ** 2 + (gy - ty) ** 2) / (2.0 * scent_sigma**2)
        )

    rodent_xy = qpos[:, :2]

    # ---- Build the figure with 3 panels and animate. ----
    fig = plt.figure(figsize=(15, 5), dpi=110)
    ax_render = fig.add_subplot(1, 3, 1)
    ax_scent = fig.add_subplot(1, 3, 2)
    ax_trace = fig.add_subplot(1, 3, 3)

    ax_render.set_axis_off()
    ax_render.set_title("3D render")
    im_render = ax_render.imshow(rgb_frames[0])

    ax_scent.set_title("Scent heatmap")
    ax_scent.set_xlim(-extent, extent)
    ax_scent.set_ylim(-extent, extent)
    ax_scent.set_aspect("equal")
    im_scent = ax_scent.imshow(
        np.zeros((grid_resolution, grid_resolution)),
        extent=(-extent, extent, -extent, extent),
        origin="lower",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )
    (line_trail_scent,) = ax_scent.plot([], [], "r-", lw=1, alpha=0.7)
    (dot_pos_scent,) = ax_scent.plot([], [], "ro", ms=5)

    ax_trace.set_title("XY trace")
    ax_trace.set_xlim(-extent, extent)
    ax_trace.set_ylim(-extent, extent)
    ax_trace.set_aspect("equal")
    ax_trace.grid(True, alpha=0.3)
    treat_active = ax_trace.scatter(
        treat_positions[:, 0],
        treat_positions[:, 1],
        c="blue",
        s=80,
        alpha=0.85,
        label="treat",
    )
    treat_collected_dots = ax_trace.scatter(
        [], [], c="grey", s=80, alpha=0.4, label="collected"
    )
    (line_trail_trace,) = ax_trace.plot([], [], "r-", lw=1.2)
    (dot_pos_trace,) = ax_trace.plot([], [], "ro", ms=6)
    ax_trace.legend(loc="upper right", fontsize=8)

    def update(fi):
        t = int(frames_idx[fi])
        im_render.set_data(rgb_frames[fi])

        active_mask = ~treat_collected[t]  # [N_treats]
        scent = (
            per_treat_grid[active_mask].sum(axis=0)
            if active_mask.any()
            else np.zeros_like(per_treat_grid[0])
        )
        if scent.max() > 0:
            scent = scent / scent.max()
        im_scent.set_data(scent)
        trail_x = rodent_xy[: t + 1, 0]
        trail_y = rodent_xy[: t + 1, 1]
        line_trail_scent.set_data(trail_x, trail_y)
        dot_pos_scent.set_data([rodent_xy[t, 0]], [rodent_xy[t, 1]])

        line_trail_trace.set_data(trail_x, trail_y)
        dot_pos_trace.set_data([rodent_xy[t, 0]], [rodent_xy[t, 1]])
        collected_mask = treat_collected[t]
        if collected_mask.any():
            treat_collected_dots.set_offsets(treat_positions[collected_mask, :2])
        else:
            treat_collected_dots.set_offsets(np.empty((0, 2)))
        return (
            im_render,
            im_scent,
            line_trail_scent,
            dot_pos_scent,
            line_trail_trace,
            dot_pos_trace,
            treat_collected_dots,
        )

    fig.tight_layout()
    ani = manim.FuncAnimation(
        fig, update, frames=n_frames, blit=False, interval=1000 / fps
    )
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ani.save(str(out_path), writer="ffmpeg", fps=fps)
    plt.close(fig)
    print(
        f"saved arena-forage 3-panel video to {out_path} ({n_frames} frames, stride={stride})"
    )
    return out_path

---

## Section 8 — Arena-forage: SCAMPER (`mode=distill_prior`)

Files: `arenaforage_scamper.h5`, `arenaforage_scamper.mp4`.


In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_SCAMPER_CHECKPOINT, ARENAFORAGE_SCAMPER_STEP)
print(f"  obs_size={af_start_state.obs.shape[-1]}, action_size={af_env_.action_size}")

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scamper.h5"
)
print(
    f"loaded qpos {af_rollout['qpos'].shape}, treat_collected {af_rollout['treat_collected'].shape}"
)

In [ ]:
af_rollout, af_states = arena_forage_rollout(
    af_start_state,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    ARENAFORAGE_NUM_STEPS,
    seed=SEED,
)
print(
    f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
    f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
)

In [ ]:
save_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scamper.h5",
    af_rollout,
    attrs={
        "kind": "arenaforage",
        "variant": "SCAMPER",
        "mode": str(af_mode),
        "checkpoint": str(ARENAFORAGE_SCAMPER_CHECKPOINT),
        "step": (
            -1 if ARENAFORAGE_SCAMPER_STEP is None else int(ARENAFORAGE_SCAMPER_STEP)
        ),
        "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
        "num_steps_actual": int(af_rollout["qpos"].shape[0]),
        "scent_sigma": float(af_base_env._config.scent_sigma),
        "spawn_radius": float(af_base_env._config.spawn_radius),
        "num_treats": int(af_base_env._config.num_treats),
        "seed": int(SEED),
    },
)

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_rollout,
    scent_sigma=float(af_base_env._config.scent_sigma),
    spawn_radius=float(af_base_env._config.spawn_radius),
    out_path=VIDEOS_DIR / "arenaforage_scamper.mp4",
)
display(Video(str(_af_video), embed=True))

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)

---

## Section 9 — Arena-forage: MIMIC-MJX (`mode=decoder_only`)

Files: `arenaforage_mimicmjx.h5`, `arenaforage_mimicmjx.mp4`.


In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_MIMICMJX_CHECKPOINT, ARENAFORAGE_MIMICMJX_STEP)

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_mimicmjx.h5"
)
print(f"loaded qpos {af_rollout['qpos'].shape}")

In [ ]:
af_rollout, af_states = arena_forage_rollout(
    af_start_state,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    ARENAFORAGE_NUM_STEPS,
    seed=SEED,
)
print(
    f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
    f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
)

In [ ]:
save_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_mimicmjx.h5",
    af_rollout,
    attrs={
        "kind": "arenaforage",
        "variant": "MIMIC-MJX",
        "mode": str(af_mode),
        "checkpoint": str(ARENAFORAGE_MIMICMJX_CHECKPOINT),
        "step": (
            -1 if ARENAFORAGE_MIMICMJX_STEP is None else int(ARENAFORAGE_MIMICMJX_STEP)
        ),
        "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
        "num_steps_actual": int(af_rollout["qpos"].shape[0]),
        "scent_sigma": float(af_base_env._config.scent_sigma),
        "spawn_radius": float(af_base_env._config.spawn_radius),
        "num_treats": int(af_base_env._config.num_treats),
        "seed": int(SEED),
    },
)

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_rollout,
    scent_sigma=float(af_base_env._config.scent_sigma),
    spawn_radius=float(af_base_env._config.spawn_radius),
    out_path=VIDEOS_DIR / "arenaforage_mimicmjx.mp4",
)
display(Video(str(_af_video), embed=True))

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)

---

## Section 10 — Arena-forage: scratch (`mode=scratch`)

Files: `arenaforage_scratch.h5`, `arenaforage_scratch.mp4`.


In [ ]:
(
    af_env_,
    af_base_env,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    af_start_state,
    af_config,
    af_mode,
) = setup_arena_forage(ARENAFORAGE_SCRATCH_CHECKPOINT, ARENAFORAGE_SCRATCH_STEP)

In [ ]:
af_rollout, af_attrs = load_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scratch.h5"
)
print(f"loaded qpos {af_rollout['qpos'].shape}")

In [ ]:
af_rollout, af_states = arena_forage_rollout(
    af_start_state,
    af_jit_step,
    af_jit_inf,
    af_inf_params,
    ARENAFORAGE_NUM_STEPS,
    seed=SEED,
)
print(
    f"qpos {af_rollout['qpos'].shape}; treats collected at end: "
    f"{int(af_rollout['treat_collected'][-1].sum())}/{af_rollout['treat_collected'].shape[1]}"
)

In [ ]:
save_arena_forage_rollout(
    ROLLOUTS_DIR / "arenaforage_scratch.h5",
    af_rollout,
    attrs={
        "kind": "arenaforage",
        "variant": "scratch",
        "mode": str(af_mode),
        "checkpoint": str(ARENAFORAGE_SCRATCH_CHECKPOINT),
        "step": (
            -1 if ARENAFORAGE_SCRATCH_STEP is None else int(ARENAFORAGE_SCRATCH_STEP)
        ),
        "num_steps_requested": int(ARENAFORAGE_NUM_STEPS),
        "num_steps_actual": int(af_rollout["qpos"].shape[0]),
        "scent_sigma": float(af_base_env._config.scent_sigma),
        "spawn_radius": float(af_base_env._config.spawn_radius),
        "num_treats": int(af_base_env._config.num_treats),
        "seed": int(SEED),
    },
)

In [ ]:
_af_video = render_arena_forage_3panel(
    af_base_env,
    af_rollout,
    scent_sigma=float(af_base_env._config.scent_sigma),
    spawn_radius=float(af_base_env._config.spawn_radius),
    out_path=VIDEOS_DIR / "arenaforage_scratch.mp4",
)
display(Video(str(_af_video), embed=True))

In [ ]:
release(
    "af_env_",
    "af_base_env",
    "af_jit_step",
    "af_jit_inf",
    "af_inf_params",
    "af_start_state",
    "af_config",
    "af_mode",
    "af_rollout",
    "af_states",
    "af_attrs",
    "_af_video",
)